# 모델 검증 및 평가

## 가. 분류 모델 평가 지표
### 1. **Confusion Matrix (혼돈 행렬, 오차 행렬)** :
#### 가) **개념** : 분류 모델의 예측 성능을 측정하기 위해 실제 정답과 모델이 예측한 값을 비교하여 표현한 2×2 행렬
#### 나) **구조** :
- **TP (True Positive)** : 실제 True를 True로 바르게 예측
- **TN (True Negative)** : 실제 False를 False로 바르게 예측
- **FP (False Positive)** : 실제 False를 True로 잘못 예측 (1종 오류)
- **FN (False Negative)** : 실제 True를 False로 잘못 예측 (2종 오류)

### 2. **정확도 (Accuracy)** :
#### 가) **개념** : 전체 예측 데이터 건수 중, 모델이 실제 정답을 맞힌 건수의 비율
#### 나) **수식** : $Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$
#### 다) **한계** : 데이터 불균형 상태(예: 암 환자 5%, 정상 95%)에서는 모델이 아무 연산 없이 "전부 정상"이라고만 예측해도 정확도가 95%가 나오는 수학적 왜곡 발생

### 3. **재현율 (Recall)** :
#### 가) **개념** : 실제 정답이 Positive(양성)인 것들 중, 모델이 Positive라고 맞힌 비율 (실제 정답 중에 정확히 맞춘 비율)
#### 나) **수식** : $Recall = \frac{TP}{TP + FN}$

### 4. **정밀도 (Precision)** :
#### 가) **개념** : 모델이 Positive(양성)라고 예측한 것들 중, 실제 정답이 Positive인 비율 (정답으로 예측한 것 중에서 실제로 맞춘 비율)
#### 나) **수식** : $Precision = \frac{TP}{TP + FP}$

### 5. **F1-Score (조화평균)** :
#### 가) **개념** : 정밀도(Precision)와 재현율(Recall)의 조화평균
#### 나) **수식** : $F1\text{-}Score = 2 \times \frac{Precision \times Recall}{Precision + Recall}$
#### 다) **왜 산술평균이 아닌 조화평균을 쓰는가?** :
- 어느 한쪽(정밀도 또는 재현율)이 0일 때 산술평균은 0.5(50%)가 되지만, 조화평균은 0이 되어 불균형의 왜곡 현상을 완벽히 차단함

In [1]:
# [실습] 사이킷런을 이용한 분류 평가 지표 확인
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# 데이터 로드 및 모델 학습
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(cancer.data, cancer.target, test_size=0.2, random_state=0)
dt = DecisionTreeClassifier(random_state=0)
dt.fit(X_train, y_train)
pred = dt.predict(X_test)

# 평가 지표 계산
print("Confusion Matrix:\n", confusion_matrix(y_test, pred))
print(f"Accuracy: {accuracy_score(y_test, pred):.4f}")
print(f"Precision: {precision_score(y_test, pred):.4f}")
print(f"Recall: {recall_score(y_test, pred):.4f}")
print(f"F1-Score: {f1_score(y_test, pred):.4f}")

Confusion Matrix:
 [[44  3]
 [ 7 60]]
Accuracy: 0.9123
Precision: 0.9524
Recall: 0.8955
F1-Score: 0.9231


## 나. 특성 중요도 (Feature Importance)
### 1. **개념** : 학습이 완료된 모델이 예측을 수행할 때, 어떤 특성(feature)을 다른 특성보다 더 중요하게 정답의 근거로 사용했는지 알려주는 지표
### 2. **용도** : 블랙박스인 모델의 내부를 들여다보고 "이 모델은 무엇을 근거로 이런 판단을 내렸을까?"라는 질문에 대한 단서를 얻음
### 3. **특성 중요도의 종류** :
- **내장 특성 중요도 (학습용 데이터 기준)** : 학습 과정에서 각 특성이 노드 분할에 기여한 평균 불순도 감소량을 기반으로 측정
- **순열 특성 중요도 (평가용 데이터 기준)** : 학습을 마친 모델에 평가 데이터를 넣고, 특정 컬럼의 데이터를 무작위로 섞었을 때 모델 성능이 하락하는 정도를 측정
### 4. **과적합 유무 판별** :
- **일반화 성공** : 내장 중요도와 순열 중요도의 상위권 특성 순위가 유사하게 유지됨
- **과적합된 불량 모델** : 학습 데이터에서는 중요도가 높았으나, 평가 데이터(순열 중요도) 검증에서는 기여도가 폭락하거나 음수(-)로 떨어짐

In [2]:
# [실습] 내장 특성 중요도와 순열 특성 중요도 확인
from sklearn.inspection import permutation_importance
import pandas as pd

# 내장 특성 중요도
feature_importances = pd.Series(dt.feature_importances_, index=cancer.feature_names)
print("[내장 특성 중요도 상위 5개]")
print(feature_importances.sort_values(ascending=False).head(5))

# 순열 특성 중요도 (평가용 데이터 기준)
result = permutation_importance(dt, X_test, y_test, n_repeats=10, random_state=0)
perm_importances = pd.Series(result.importances_mean, index=cancer.feature_names)
print("\n[순열 특성 중요도 상위 5개]")
print(perm_importances.sort_values(ascending=False).head(5))

[내장 특성 중요도 상위 5개]
worst concave points    0.692835
worst area              0.135911
mean smoothness         0.038941
mean perimeter          0.022554
mean symmetry           0.016482
dtype: float64

[순열 특성 중요도 상위 5개]
worst area         0.282456
worst perimeter    0.227193
worst concavity    0.161404
area error         0.131579
mean concavity     0.030702
dtype: float64


## 다. 회귀 모델 평가 지표
### 1. **평균 절대 오차 (MAE: Mean Absolute Error)** :
- 실제 값과 예측 값의 차이의 절대값 평균
### 2. **평균 제곱 오차 (MSE: Mean Squared Error)** :
- 실제 값과 예측 값의 차이의 제곱을 모두 더하여 평균을 구함 ($MSE = \frac{\sum(y - \hat{y})^2}{n}$)
### 3. **평균 제곱근 오차 (RMSE: Root Mean Squared Error)** :
- MSE에 제곱근(Root)을 씌운 값 ($RMSE = \sqrt{\frac{\sum(y - \hat{y})^2}{n}}$)

In [3]:
# [실습] 회귀 평가 지표 (MAE, MSE, RMSE)
from sklearn.datasets import fetch_california_housing
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error

housing = fetch_california_housing()
X_r, y_r = housing.data, housing.target
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_r, y_r, test_size=0.2, random_state=0)

dt_reg = DecisionTreeRegressor(random_state=0)
dt_reg.fit(X_train_r, y_train_r)
pred_reg = dt_reg.predict(X_test_r)

print(f"MAE: {mean_absolute_error(y_test_r, pred_reg):.4f}")
print(f"MSE: {mean_squared_error(y_test_r, pred_reg):.4f}")
print(f"RMSE: {root_mean_squared_error(y_test_r, pred_reg):.4f}")

MAE: 0.4695
MSE: 0.5347
RMSE: 0.7312
